# 🛰️ SatQuery AI — GPU Backend Server (Universal: Kaggle & Google Colab)

This notebook runs the **full production backend with real deep-learning models** on **Kaggle's or Google Colab's free 16GB GPU** (NVIDIA Tesla T4/P100).

> **Prerequisites:**
> - **If on Kaggle:** In the right sidebar, set **Accelerator** to **GPU T4 x2** (or P100) and **Internet** to **ON**.
> - **If on Google Colab:** Go to **Runtime** → **Change runtime type** → select **T4 GPU**.

---

### Step 1: Verify NVIDIA GPU & Clone Project Codebase

In [ ]:
!nvidia-smi
import os
# Auto-detect runtime: Google Colab (/content) vs Kaggle (/kaggle/working)
WORKDIR = "/content" if os.path.exists("/content") else "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
os.chdir(WORKDIR)

repo_path = os.path.join(WORKDIR, "novex")
if os.path.exists(repo_path):
    os.chdir(repo_path)
    !git pull
else:
    !git clone https://github.com/sanskar620/novex.git
    os.chdir(repo_path)

print(f"\n✅ Environment: {WORKDIR}")
print(f"✅ Repository ready at: {repo_path}")

### Step 2: Install Geospatial & ML Dependencies

In [ ]:
# System GIS packages & Cloudflare Tunnel binary
!apt-get update -qq && apt-get install -y -qq gdal-bin libgdal-dev libgeos-dev libproj-dev > /dev/null
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Install all backend dependencies (including structlog, celery, minio, psycopg2)
!pip install -q structlog "python-jose[cryptography]" celery minio psycopg2-binary redis
!pip install -q fastapi uvicorn pydantic pydantic-settings sqlalchemy alembic python-multipart
!pip install -q numpy pillow matplotlib rasterio shapely geopandas pyproj reportlab httpx
!pip install -q "transformers>=4.44.0,<=4.46.2" accelerate timm
!pip install -q --no-deps -e backend/

print("✅ All dependencies installed successfully!")

### Step 3: Model Selection & Pre-Downloading
We load two real specialist models:
1. **Vision-Language Model:** `OpenGVLab/InternVL2-1B` (Remote sensing VQA, scene captioning, spatial bounding boxes)
2. **Task Planning Agent:** `Qwen/Qwen2.5-1.5B-Instruct` (Interprets user queries and orchestrates spatial pipelines)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

# Universal descriptor with getter AND setter to support Qwen2 & InternVL
class _TiedKeysDescriptor:
    def __get__(self, obj, cls=None):
        if obj is None:
            return self
        return getattr(obj, "_all_tied_keys_store", {})
    def __set__(self, obj, val):
        obj._all_tied_keys_store = val

nn.Module.all_tied_weights_keys = _TiedKeysDescriptor()

INTERNVL_MODEL = "OpenGVLab/InternVL2-1B"
QWEN_AGENT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"📥 [1/2] Pre-downloading Vision-Language Model: {INTERNVL_MODEL} ...")
AutoTokenizer.from_pretrained(INTERNVL_MODEL, trust_remote_code=True)
AutoModel.from_pretrained(INTERNVL_MODEL, trust_remote_code=True, torch_dtype=torch.float16)

print(f"📥 [2/2] Pre-downloading Agent Planner Model: {QWEN_AGENT_MODEL} ...")
AutoTokenizer.from_pretrained(QWEN_AGENT_MODEL)
AutoModelForCausalLM.from_pretrained(QWEN_AGENT_MODEL, torch_dtype=torch.float16)

print("\n✅ Both real neural models downloaded and cached in GPU memory successfully!")

### Step 4: Configure Environment Variables (Real GPU Mode Active)

In [ ]:
import os

os.environ["DEMO_MODE"] = "false"
os.environ["ENVIRONMENT"] = "production"
os.environ["DEVICE"] = "cuda"
os.environ["INTERNVL_MODEL_PATH"] = INTERNVL_MODEL
os.environ["AGENT_MODEL_NAME"] = "Qwen2.5-1.5B"
os.environ["AGENT_MODEL_PATH"] = QWEN_AGENT_MODEL
os.environ["CORS_ALLOWED_ORIGINS"] = "https://novex26.vercel.app,https://novex-teal.vercel.app,http://localhost:3000"

print("✅ Configuration Active:")
print(f" • DEMO_MODE: {os.environ['DEMO_MODE']} (Real Neural Inference Enabled)")
print(f" • DEVICE: {os.environ['DEVICE']} (CUDA Hardware Acceleration)")
print(f" • Vision Model: {INTERNVL_MODEL}")
print(f" • Agent Model: {QWEN_AGENT_MODEL}")

### Step 5: Start Public Cloudflare Tunnel & Launch Backend Server
Run this cell. It will print a public **`https://xxxx.trycloudflare.com`** URL.
Copy that URL and add it to your Vercel project!

In [ ]:
import subprocess
import threading
import time
import re
import os

def monitor_tunnel():
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print("\n" + "="*70)
            print("🚀 YOUR LIVE GPU BACKEND IS ONLINE!")
            print(f"👉 Public Backend URL: {url}")
            print(f"👉 Set this in Vercel (NEXT_PUBLIC_GPU_API_BASE_URL): {url}/api/v1")
            print("="*70 + "\n")

tunnel_thread = threading.Thread(target=monitor_tunnel, daemon=True)
tunnel_thread.start()
time.sleep(3)

# Start FastAPI Uvicorn Server in backend directory
WORKDIR = "/content" if os.path.exists("/content") else "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
backend_dir = os.path.join(WORKDIR, "novex", "backend")
if os.path.exists(backend_dir):
    os.chdir(backend_dir)
elif os.path.exists("backend"):
    os.chdir("backend")

!python -m uvicorn app.main:app --host 0.0.0.0 --port 8000